## Prepare the runtime


In [ ]:
import json
import time
from typing import Optional

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

def sync_gpu():
    if torch.cuda.is_available():
        torch.cuda.synchronize()


## Load the tokenizer and model


In [ ]:
MODEL_ID = "Qwen/Qwen3-4B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
).eval()

print(f"Loaded {MODEL_ID}")


## Prepare the model input


In [ ]:
def prepare_inputs(prompt: str):
    # TODO 1: create system and user messages, then render the generation prompt.
    messages = [
            {"role": "system", "content": "Answer the Question"},
            {"role": "user", "content": prompt}
        ]
    
    inputs = tokenizer.apply_chat_template(
        messages, 
        return_tensors="pt", 
        return_dict=True
    )
    assert messages, "Add the prompt messages"
    assert inputs is not None, "Render the prompt with the tokenizer"
    return {key: value.to(model.device) for key, value in inputs.items()}


## Add temperature and top-k sampling


In [ ]:
def select_next_token(
    logits: torch.Tensor,
    temperature: float,
    top_k: Optional[int] = None,
) -> torch.Tensor:
    if temperature == 0:
        return torch.argmax(logits, dim=-1, keepdim=True)

    # TODO 2: scale logits, apply the top-k cutoff, and sample one token.
    scaled_logits = logits / temperature
    if top_k is not None:
        top_values,_ = torch.topk(scaled_logits,min(top_k,scaled_logits.size(-1)))
        cutoff = top_values[...,-1,None]
        scaled_logits = torch.where(scaled_logits < cutoff , float('-inf'),scaled_logits)
        

    probabilities = torch.softmax(scaled_logits, dim=-1)
    next_token = torch.multinomial(probabilities, num_samples=1)
    assert next_token is not None, "Sample one token"
    return next_token


## Build generation without KV cache


In [ ]:
@torch.inference_mode()
def generate_uncached(
    prompt: str,
    max_new_tokens: int = 32,
    temperature: float = 0.0,
    top_k: Optional[int] = None,
) -> dict:
    inputs = prepare_inputs(prompt)
    generated_ids = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]
    prompt_length = generated_ids.shape[1]
    first_token_at = None

    sync_gpu()
    started_at = time.perf_counter()
    for _ in range(max_new_tokens):
        outputs = model(
            input_ids=generated_ids,
            attention_mask=attention_mask,
            use_cache=False,
        )

        # TODO 3: select and append the next token, extend the mask, and stop on EOS.
        next_token = select_next_token(
            outputs.logits[:, -1, :],
            temperature,
            top_k,
        )
        if first_token_at is None:
            sync_gpu()
            first_token_at = time.perf_counter()

        generated_ids = torch.cat([generated_ids, next_token], dim=-1)
        attention_mask = torch.cat([attention_mask, torch.ones_like(next_token)], dim=-1)
        if next_token.item() == tokenizer.eos_token_id:
            break

    sync_gpu()
    finished_at = time.perf_counter()
    completion_ids = generated_ids[0, prompt_length:]
    return {
        "text": tokenizer.decode(completion_ids, skip_special_tokens=True),
        "token_ids": completion_ids.tolist(),
        "input_tokens": prompt_length,
        "ttft_ms": (first_token_at - started_at) * 1000,
        "latency_ms": (finished_at - started_at) * 1000,
    }


## Add KV-cached generation


In [ ]:
@torch.inference_mode()
def generate_cached(
    prompt: str,
    max_new_tokens: int = 32,
    temperature: float = 0.0,
    top_k: Optional[int] = None,
) -> dict:
    inputs = prepare_inputs(prompt)
    generated_ids = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]
    prompt_length = generated_ids.shape[1]
    first_token_at = None

    sync_gpu()
    started_at = time.perf_counter()
    outputs = model(
        input_ids=generated_ids,
        attention_mask=attention_mask,
        use_cache=True,
    )
    past_key_values = outputs.past_key_values

    for _ in range(max_new_tokens):
        # TODO 4: select the next token and reuse past_key_values for the next model call.
        next_token = select_next_token(
            outputs.logits[:, -1, :],
            temperature,
            top_k,
        )       
        
        assert next_token is not None, "Select the next token"
        if first_token_at is None:
            sync_gpu()
            first_token_at = time.perf_counter()

        generated_ids = torch.cat([generated_ids, next_token], dim=1)
        if next_token.item() == tokenizer.eos_token_id:
            break

        attention_mask = torch.cat([
            attention_mask,
            torch.ones((1, 1), dtype=attention_mask.dtype, device=model.device),
        ], dim=1)
        outputs = model(
            input_ids=generated_ids,
            attention_mask=attention_mask,
            use_cache=False,
        )
        past_key_values = outputs.past_key_values

    sync_gpu()
    finished_at = time.perf_counter()
    completion_ids = generated_ids[0, prompt_length:]
    return {
        "text": tokenizer.decode(completion_ids, skip_special_tokens=True),
        "token_ids": completion_ids.tolist(),
        "input_tokens": prompt_length,
        "ttft_ms": (first_token_at - started_at) * 1000,
        "latency_ms": (finished_at - started_at) * 1000,
    }


## Calculate serving metrics


In [ ]:
def summarize_result(result: dict, used_kv_cache: bool) -> dict:
    # TODO 5: calculate output token count and output tokens per second.
    output_tokens = len(result['token_ids'])
    output_tokens_per_second = output_tokens / (result['latency_ms']/1000)

    return {
        "text": result["text"],
        "ttft_ms": round(result["ttft_ms"], 2),
        "latency_ms": round(result["latency_ms"], 2),
        "input_tokens": result["input_tokens"],
        "output_tokens": output_tokens,
        "output_tokens_per_second": round(output_tokens_per_second, 2),
        "used_kv_cache": used_kv_cache,
    }


## Compare generation with and without KV cache


In [ ]:
BENCHMARK_PROMPT = "Explain why the sky looks blue in two sentences."

print("Warming up the model", flush=True)
generate_cached("Say hello in one sentence.", max_new_tokens=2)

print("Running without KV cache", flush=True)
uncached_result = generate_uncached(BENCHMARK_PROMPT, max_new_tokens=24)
print("Running with KV cache", flush=True)
cached_result = generate_cached(BENCHMARK_PROMPT, max_new_tokens=24)

comparison = {
    "outputs_match": uncached_result["token_ids"] == cached_result["token_ids"],
    "without_cache": summarize_result(uncached_result, used_kv_cache=False),
    "with_cache": summarize_result(cached_result, used_kv_cache=True),
}
print("__TT_COMPARISON__=" + json.dumps(comparison), flush=True)


## Stream generated text


In [ ]:
@torch.inference_mode()
def generate_stream(
    prompt: str,
    max_new_tokens: int = 32,
    temperature: float = 0.0,
    top_k: Optional[int] = None,
):
    inputs = prepare_inputs(prompt)
    generated_ids = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]
    prompt_length = generated_ids.shape[1]

    outputs = model(
        input_ids=generated_ids,
        attention_mask=attention_mask,
        use_cache=True,
    )
    past_key_values = outputs.past_key_values
    previous_text = ""

    for _ in range(max_new_tokens):
        # TODO 6: select the next token and yield only the newly decoded text.
        next_token = select_next_token(
            outputs.logits[:, -1, :],
            temperature,
            top_k,
        )         
        assert next_token is not None, "Select the next token"
        generated_ids = torch.cat([generated_ids, next_token], dim=1)

        completion_ids = generated_ids[0, prompt_length:]
        current_text = tokenizer.decode(completion_ids, skip_special_tokens=True)
        chunk = current_text[len(previous_text):]
        if chunk:
            yield chunk
        previous_text = current_text

        if next_token.item() == tokenizer.eos_token_id:
            break

        attention_mask = torch.cat([
            attention_mask,
            torch.ones(
                (1, 1),
                dtype=attention_mask.dtype,
                device=model.device,
            ),
        ], dim=1)
        outputs = model(
            input_ids=next_token,
            attention_mask=attention_mask,
            past_key_values=past_key_values,
            use_cache=True,
        )
        past_key_values = outputs.past_key_values


## Define the request and response


In [ ]:
from typing import Optional
from pydantic import BaseModel, Field

class GenerateRequest(BaseModel):
    prompt: str = Field(..., min_length=1, description="The input text prompt.")
    max_new_tokens: int = Field(default=32, ge=1, le=512, description="Maximum tokens to generate.")
    temperature: float = Field(default=0.0, ge=0.0, le=2.0, description="Sampling temperature.")
    top_k: Optional[int] = Field(default=None, ge=1, le=100, description="Top-k token cutoff.")
class GenerateResponse(BaseModel):
    text: str = Field(..., description="The decoded generated completion.")
    ttft_ms: float = Field(..., description="Time to first token in milliseconds.")
    latency_ms: float = Field(..., description="Total execution time in milliseconds.")
    input_tokens: int = Field(..., description="Prompt token count.")
    output_tokens: int = Field(..., description="Output token count.")
    output_tokens_per_second: float = Field(..., description="Throughput in tokens per second.")
    used_kv_cache: bool = Field(..., description="Indicates if KV caching was active.")


## Create the server endpoints


In [ ]:
from fastapi import FastAPI
from fastapi.responses import StreamingResponse

app = FastAPI(title="tinyinference")

@app.get("/health")
def health():
    return {"status": "ok"}

# TODO 8: implement the measured generation and streaming endpoints.
@app.post("/generate", response_model=GenerateResponse)
def generate_text(request: GenerateRequest):
    result = generate_cached(
        prompt=request.prompt,
        max_new_tokens=request.max_new_tokens,
        temperature=request.temperature,
        top_k=request.top_k,
    )
    output_tokens = len(result["token_ids"])
    latency_sec = result["latency_ms"] / 1000.0
    output_tokens_per_second = output_tokens / latency_sec if latency_sec > 0 else 0.0
    used_kv_cache = True

    return {
        "text": result["text"],
        "ttft_ms": round(result["ttft_ms"], 2),
        "latency_ms": round(result["latency_ms"], 2),
        "input_tokens": result["input_tokens"],
        "output_tokens": output_tokens,
        "output_tokens_per_second": round(output_tokens_per_second, 2),
        "used_kv_cache": used_kv_cache,
    }

@app.post("/stream")
def stream_text(request: GenerateRequest):
    return StreamingResponse(
        generate_stream(
            prompt=request.prompt,
            max_new_tokens=request.max_new_tokens,
            temperature=request.temperature,
            top_k=request.top_k,
        ),
        media_type="text/plain"
    )


## Test the inference server


In [ ]:
from fastapi.testclient import TestClient

client = TestClient(app)

# TODO 9: call the health and generation endpoints and verify their responses.
health_response = client.get("/health")
generation_response = client.post("/generate", json={"prompt": "Hello"})

assert health_response.status_code == 200
assert health_response.json() == {"status": "ok"}
assert generation_response.status_code == 200
assert generation_response.json()["output_tokens"] > 0

print(json.dumps(generation_response.json(), indent=2))
